# 01 — Data Cleaning & Merging
**PrimeTrade.ai Data Science Assignment**

This notebook:
1. Loads both raw datasets
2. Cleans and standardizes each
3. Merges them on date
4. Saves a clean master dataframe for all downstream analysis


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

Libraries loaded ✅


## 1. Load Raw Data

In [2]:
# ── Load Fear & Greed Index ──────────────────────────────────────────────────
fg_raw = pd.read_csv('data/fear_greed_index.csv')
print(f'Fear/Greed shape: {fg_raw.shape}')
print(fg_raw.head(3))

Fear/Greed shape: (2644, 4)
    timestamp  value classification        date
0  1517463000     30           Fear  2018-02-01
1  1517549400     15   Extreme Fear  2018-02-02
2  1517635800     40           Fear  2018-02-03


In [3]:
# ── Load Historical Trader Data ──────────────────────────────────────────────
trader_raw = pd.read_csv('data/historical_data.csv')
print(f'Trader data shape: {trader_raw.shape}')
print(trader_raw.head(3))

Trader data shape: (211224, 16)
                                      Account  Coin  Execution Price  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   

   Size Tokens  Size USD Side     Timestamp IST  Start Position Direction  \
0       986.87   7872.16  BUY  02-12-2024 22:50        0.000000       Buy   
1        16.00    127.68  BUY  02-12-2024 22:50      986.524596       Buy   
2       144.09   1150.63  BUY  02-12-2024 22:50     1002.518996       Buy   

   Closed PnL                                   Transaction Hash     Order ID  \
0         0.0  0xec09451986a1874e3a980418412fcd0201f500c95bac...  52017706630   
1         0.0  0xec09451986a1874e3a980418412fcd0201f500c95bac...  52017706630   
2         0.0  0xec09451986a1874e3a980418412fcd0201f500c95bac...  52017706630   

   Crossed       Fee      Trade ID     Timest

## 2. Clean Fear & Greed Dataset

In [4]:
fg = fg_raw.copy()

# Parse date
fg['date'] = pd.to_datetime(fg['date'])

# Rename for clarity
fg.rename(columns={'value': 'fg_value', 'classification': 'sentiment'}, inplace=True)

# Drop timestamp column (redundant with date)
fg.drop(columns=['timestamp'], inplace=True)

# Define ordinal sentiment order
sentiment_order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
fg['sentiment'] = pd.Categorical(fg['sentiment'], categories=sentiment_order, ordered=True)

# Add numeric sentiment code (1=Extreme Fear ... 5=Extreme Greed)
fg['sentiment_code'] = fg['sentiment'].cat.codes + 1

# Broad category: Fear / Neutral / Greed
def broad_sentiment(s):
    if s in ['Extreme Fear', 'Fear']:
        return 'Fear'
    elif s == 'Neutral':
        return 'Neutral'
    else:
        return 'Greed'

fg['broad_sentiment'] = fg['sentiment'].astype(str).apply(broad_sentiment)

print('Fear/Greed cleaned ✅')
print(fg.dtypes)
print(fg['sentiment'].value_counts())

Fear/Greed cleaned ✅
fg_value                    int64
sentiment                category
date               datetime64[ns]
sentiment_code               int8
broad_sentiment            object
dtype: object
sentiment
Fear             781
Greed            633
Extreme Fear     508
Neutral          396
Extreme Greed    326
Name: count, dtype: int64


## 3. Clean Trader Dataset

In [5]:
trader = trader_raw.copy()

# ── Rename columns ────────────────────────────────────────────────────────────
trader.columns = [
    'account', 'coin', 'exec_price', 'size_tokens', 'size_usd',
    'side', 'timestamp_ist', 'start_position', 'direction',
    'closed_pnl', 'tx_hash', 'order_id', 'crossed', 'fee',
    'trade_id', 'timestamp_raw'
]

# ── Parse datetime ────────────────────────────────────────────────────────────
trader['datetime'] = pd.to_datetime(trader['timestamp_ist'], format='%d-%m-%Y %H:%M')
trader['date'] = trader['datetime'].dt.normalize()   # date only, used for merging
trader['hour'] = trader['datetime'].dt.hour
trader['month'] = trader['datetime'].dt.to_period('M')
trader['weekday'] = trader['datetime'].dt.day_name()

# ── Standardize side & direction ──────────────────────────────────────────────
trader['side'] = trader['side'].str.strip().str.upper()       # BUY / SELL
trader['direction'] = trader['direction'].str.strip()

# ── Trade type flags ──────────────────────────────────────────────────────────
trader['is_open']  = trader['direction'].str.contains('Open', case=False, na=False)
trader['is_close'] = trader['direction'].str.contains('Close', case=False, na=False)
trader['is_long']  = trader['direction'].str.contains('Long', case=False, na=False) | (trader['side'] == 'BUY')
trader['is_short'] = trader['direction'].str.contains('Short', case=False, na=False) | (trader['side'] == 'SELL')

# ── PnL-derived columns ───────────────────────────────────────────────────────
trader['pnl_net']      = trader['closed_pnl'] - trader['fee']   # net after fees
trader['is_winner']    = trader['closed_pnl'] > 0
trader['is_loser']     = trader['closed_pnl'] < 0
trader['has_closed_trade'] = trader['closed_pnl'] != 0          # opening trades have 0 PnL

# ── Size bucketing ────────────────────────────────────────────────────────────
trader['size_bucket'] = pd.cut(
    trader['size_usd'],
    bins=[0, 100, 1_000, 10_000, 100_000, float('inf')],
    labels=['Micro(<$100)', 'Small($100-1K)', 'Medium($1K-10K)', 'Large($10K-100K)', 'Whale(>$100K)']
)

print('Trader data cleaned ✅')
print(f'Shape: {trader.shape}')
print(trader.dtypes)

Trader data cleaned ✅
Shape: (211224, 30)
account                     object
coin                        object
exec_price                 float64
size_tokens                float64
size_usd                   float64
side                        object
timestamp_ist               object
start_position             float64
direction                   object
closed_pnl                 float64
tx_hash                     object
order_id                     int64
crossed                       bool
fee                        float64
trade_id                   float64
timestamp_raw              float64
datetime            datetime64[ns]
date                datetime64[ns]
hour                         int32
month                    period[M]
weekday                     object
is_open                       bool
is_close                      bool
is_long                       bool
is_short                      bool
pnl_net                    float64
is_winner                     bool
is_loser     

In [6]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
print('=== NULL CHECK ===')
print(trader.isnull().sum()[trader.isnull().sum() > 0])

print('\n=== SIDE DISTRIBUTION ===')
print(trader['side'].value_counts())

print('\n=== DIRECTION DISTRIBUTION ===')
print(trader['direction'].value_counts())

print('\n=== CLOSING TRADES PnL STATS ===')
closed = trader[trader['has_closed_trade']]
print(closed['closed_pnl'].describe())
print(f'Win rate: {closed["is_winner"].mean():.2%}')

=== NULL CHECK ===


size_bucket    43
dtype: int64



=== SIDE DISTRIBUTION ===
side
SELL    108528
BUY     102696
Name: count, dtype: int64

=== DIRECTION DISTRIBUTION ===
direction
Open Long                    49895
Close Long                   48678
Open Short                   39741
Close Short                  36013
Sell                         19902
Buy                          16716
Spot Dust Conversion           142
Short > Long                    70
Long > Short                    57
Auto-Deleveraging                8
Liquidated Isolated Short        1
Settlement                       1
Name: count, dtype: int64

=== CLOSING TRADES PnL STATS ===
count    104408.000000
mean         98.622318
std        1305.489556
min     -117990.104100
25%           0.413069
50%           6.055500
75%          38.197947
max      135329.090100
Name: closed_pnl, dtype: float64
Win rate: 83.20%


## 4. Merge Datasets

In [7]:
# Merge trader data with fear/greed on date
merged = trader.merge(
    fg[['date', 'fg_value', 'sentiment', 'sentiment_code', 'broad_sentiment']],
    on='date',
    how='left'
)

# Check merge quality
matched   = merged['sentiment'].notna().sum()
unmatched = merged['sentiment'].isna().sum()
print(f'Rows matched with sentiment: {matched:,}  ({matched/len(merged):.2%})')
print(f'Rows unmatched:              {unmatched:,}  ({unmatched/len(merged):.2%})')

# Drop unmatched rows (only 1 day missing — negligible)
merged.dropna(subset=['sentiment'], inplace=True)
merged.reset_index(drop=True, inplace=True)

print(f'\nFinal merged shape: {merged.shape}')
merged.head(3)

Rows matched with sentiment: 211,218  (100.00%)
Rows unmatched:              6  (0.00%)

Final merged shape: (211218, 34)


,account,coin,exec_price,size_tokens,size_usd,side,timestamp_ist,start_position,direction,closed_pnl,...,is_short,pnl_net,is_winner,is_loser,has_closed_trade,size_bucket,fg_value,sentiment,sentiment_code,broad_sentiment
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,...,False,-0.345404,False,False,False,Medium($1K-10K),80.0,Extreme Greed,5.0,Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,...,False,-0.005600,False,False,False,Small($100-1K),80.0,Extreme Greed,5.0,Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,...,False,-0.050431,False,False,False,Medium($1K-10K),80.0,Extreme Greed,5.0,Greed


## 5. Save Clean Data

In [8]:
import os
os.makedirs('outputs', exist_ok=True)

# Save master merged file
merged.to_csv('outputs/merged_clean.csv', index=False)

# Save cleaned fear/greed separately
fg.to_csv('outputs/fear_greed_clean.csv', index=False)

print('✅ Saved:')
print('  outputs/merged_clean.csv')
print('  outputs/fear_greed_clean.csv')
print(f'\nMerged dataset: {merged.shape[0]:,} rows × {merged.shape[1]} columns')

✅ Saved:
  outputs/merged_clean.csv
  outputs/fear_greed_clean.csv

Merged dataset: 211,218 rows × 34 columns


## Summary

| Dataset | Rows | Key Date Range |
|---|---|---|
| Fear & Greed Index | 2,644 | Feb 2018 – May 2025 |
| Trader Data | 211,224 | Jan 2024 – Dec 2024 |
| **Merged (clean)** | **~211,200** | **Jan 2024 – Dec 2024** |

**Key facts discovered:**
- 32 unique trader accounts
- 246 unique coins traded (HYPE dominates at 68K trades)
- 50.6% of rows have non-zero PnL (i.e., are closing trades)
- Overall win rate on closed trades: ~83%
- 479 out of 480 trading days matched with sentiment data
